# AI/ML Engineer TP Test - Angel Yanciel Payano Lanfranco
Thank you for the opportunity. I will be developing each core functionality of the requested AI system within this Jupyter Notebook. This approach allows you to review the underlying logic and see exactly how the backend data processing pipelines are structured before they are integrated into the final user interface.

For the final deliverable, I will deploy the solution as a web application using Railway, allowing you to test and interact with the tool directly. I will also provide the GitHub repository containing the complete, production-ready codebase.

Since you want the system to work without having to change the code if there is a change on the business logic or contract, I am going to use the contracts.csv file as the basis of the analysis (Bonus challenge). The system is going to consist of a drag and drop app where the user will first drag and drop the contract file and then do the same with both the billing and timesheet files. Also, we will have different sliders to enhance the user experience and allow for a better analysis. At the end, the AI model, which is going to be gpt-5.4 (OPEN AI) in this case, will create an analysis based on each ERROR case and also one based on the whole ERROR cases dataset, so you can have a general analysis and a personalized analysis, providing both sides of actionable recommendations.



In [24]:
import pandas as pd

# Load datasets using relative paths to ensure portability across environments
df_contracts = pd.read_csv("contracts.csv")
df_timesheet = pd.read_csv("timesheet.csv")
df_billing = pd.read_csv("billing.csv")

# Display the number of records processed for each dataset
print(f"Contracts dataset processed: {df_contracts.shape[0]} project rules.")
print(f"Timesheet dataset processed: {df_timesheet.shape[0]} job records.")
print(f"Billing dataset processed: {df_billing.shape[0]} billing records.")

# Preview the data provided
display(df_contracts.head(10))
display(df_timesheet.head(10))
display(df_billing.head(10))

Contracts dataset processed: 3 project rules.
Timesheet dataset processed: 5 job records.
Billing dataset processed: 5 billing records.


,Project,Rate_per_Hour,Max_Hours_Per_Week
0,A,20,40
1,B,25,40
2,C,30,35


,Employee_ID,Employee_Name,Project,Hours_Worked
0,101,Alice,A,40
1,102,Bob,A,38
2,103,Charlie,B,45
3,104,Diana,B,42
4,105,Eve,C,36


,Employee_ID,Project,Hours_Billed,Rate_Charged
0,101,A,40,20
1,102,A,40,22
2,103,B,50,25
3,104,B,42,25
4,105,C,36,28


In [25]:
import numpy as np
import pandas as pd

# First, we are going to join the timesheet with the contract rules using the 'Project' column. Starting to merge the data to make a cohesive information.
df_step1 = pd.merge(df_timesheet, df_contracts, on="Project", how="left")

# Next, we join the result with the actual billed data using 'Employee_ID' and 'Project'
df_master = pd.merge(df_step1, df_billing, on=["Employee_ID", "Project"], how="left")

# 2. Validation Logic (The Rule Engine)
df_master['STATUS'] = 'OK'
df_master['DISCREPANCY_REASON'] = ''

# Now, we are going to create the rules.

# Rule A: Rate Mismatch (The charged rate differs from the contract rate)
mask_rate = df_master['Rate_Charged'] != df_master['Rate_per_Hour']
df_master.loc[mask_rate, 'STATUS'] = 'ERROR'
df_master.loc[mask_rate, 'DISCREPANCY_REASON'] += 'Rate Mismatch. '

# Rule B: Overbilling (Contract) - Billed hours exceed the maximum allowed per project
mask_max_hours = df_master['Hours_Billed'] > df_master['Max_Hours_Per_Week']
df_master.loc[mask_max_hours, 'STATUS'] = 'ERROR'
df_master.loc[mask_max_hours, 'DISCREPANCY_REASON'] += 'Exceeds Contract Max Hours. '

# Rule C: Overbilling (Actual) - Billed hours exceed the hours reported by the employee
mask_worked_hours = df_master['Hours_Billed'] > df_master['Hours_Worked']
df_master.loc[mask_worked_hours, 'STATUS'] = 'ERROR'
df_master.loc[mask_worked_hours, 'DISCREPANCY_REASON'] += 'Billed more than worked. '

# Rule D: Missing Hours (Underbilling) - The employee worked more hours than what was billed
mask_missing_hours = df_master['Hours_Billed'] < df_master['Hours_Worked']
df_master.loc[mask_missing_hours, 'STATUS'] = 'ERROR'
df_master.loc[mask_missing_hours, 'DISCREPANCY_REASON'] += 'Missing hours (Underbilling). '

# Clean up trailing white spaces in the strings
df_master['DISCREPANCY_REASON'] = df_master['DISCREPANCY_REASON'].str.strip()

# 3. Results Presentation in the Notebook
print(f"Total records processed: {len(df_master)}")
print(f"Total errors found: {len(df_master[df_master['STATUS'] == 'ERROR'])}")
print(f"Total valid records: {len(df_master[df_master['STATUS'] == 'OK'])}")

# Isolate the errors dataset to be sent to the OpenAI API later
df_errors = df_master[df_master['STATUS'] == 'ERROR'].copy()

print("\nValidation results (expected vs actual billing values, clean dataset output):")
# We remove the column width limit to show the full text of the discrepancy found
pd.set_option('display.max_colwidth', None)
# Displaying the relevant columns for the complete dataset
columns_to_display = ['Employee_Name', 'Project', 'Hours_Worked', 'Hours_Billed', 'Rate_per_Hour', 'Rate_Charged', 'STATUS', 'DISCREPANCY_REASON']
display(df_master[columns_to_display])

Total records processed: 5
Total errors found: 4
Total valid records: 1

Validation results (expected vs actual billing values, clean dataset output):


,Employee_Name,Project,Hours_Worked,Hours_Billed,Rate_per_Hour,Rate_Charged,STATUS,DISCREPANCY_REASON
0,Alice,A,40,40,20,20,OK,
1,Bob,A,38,40,20,22,ERROR,Rate Mismatch. Billed more than worked.
2,Charlie,B,45,50,25,25,ERROR,Exceeds Contract Max Hours. Billed more than worked.
3,Diana,B,42,42,25,25,ERROR,Exceeds Contract Max Hours.
4,Eve,C,36,36,30,28,ERROR,Rate Mismatch. Exceeds Contract Max Hours.


In [26]:
# The errors found are what is really of our interest, so here I show the dataframe with that information in particular.
# As we can see, we got 4 errors out of 5 
pd.set_option('display.max_colwidth', None)
df_errors

,Employee_ID,Employee_Name,Project,Hours_Worked,Rate_per_Hour,Max_Hours_Per_Week,Hours_Billed,Rate_Charged,STATUS,DISCREPANCY_REASON
1,102,Bob,A,38,20,40,40,22,ERROR,Rate Mismatch. Billed more than worked.
2,103,Charlie,B,45,25,40,50,25,ERROR,Exceeds Contract Max Hours. Billed more than worked.
3,104,Diana,B,42,25,40,42,25,ERROR,Exceeds Contract Max Hours.
4,105,Eve,C,36,30,35,36,28,ERROR,Rate Mismatch. Exceeds Contract Max Hours.


# Automated Workflow

In [27]:
import pandas as pd
import numpy as np

def automated_billing_validation(timesheet_path, contracts_path, billing_path):
    """
    Automates the end-to-end data validation workflow.
    Simulates the process that will occur when files are uploaded to the web interface.
    """
    try:
        # 1. Ingestion: Load datasets
        df_timesheet = pd.read_csv(timesheet_path)
        df_contracts = pd.read_csv(contracts_path)
        df_billing = pd.read_csv(billing_path)
        
        # 2. Processing: Master DataFrame Creation
        df_step1 = pd.merge(df_timesheet, df_contracts, on="Project", how="left")
        df_master = pd.merge(df_step1, df_billing, on=["Employee_ID", "Project"], how="left")
        
        # 3. Automation: Rule Engine Application
        df_master['STATUS'] = 'OK'
        df_master['DISCREPANCY_REASON'] = ''
        
        # Rule A: Rate Mismatch
        mask_rate = df_master['Rate_Charged'] != df_master['Rate_per_Hour']
        df_master.loc[mask_rate, 'STATUS'] = 'ERROR'
        df_master.loc[mask_rate, 'DISCREPANCY_REASON'] += 'Rate Mismatch. '
        
        # Rule B: Contract Overbilling
        mask_max_hours = df_master['Hours_Billed'] > df_master['Max_Hours_Per_Week']
        df_master.loc[mask_max_hours, 'STATUS'] = 'ERROR'
        df_master.loc[mask_max_hours, 'DISCREPANCY_REASON'] += 'Exceeds Contract Max Hours. '
        
        # Rule C: Actual Overbilling
        mask_worked_hours = df_master['Hours_Billed'] > df_master['Hours_Worked']
        df_master.loc[mask_worked_hours, 'STATUS'] = 'ERROR'
        df_master.loc[mask_worked_hours, 'DISCREPANCY_REASON'] += 'Billed more than worked. '
        
        # Rule D: Missing Hours (Underbilling)
        mask_missing_hours = df_master['Hours_Billed'] < df_master['Hours_Worked']
        df_master.loc[mask_missing_hours, 'STATUS'] = 'ERROR'
        df_master.loc[mask_missing_hours, 'DISCREPANCY_REASON'] += 'Missing hours (Underbilling). '
        
        df_master['DISCREPANCY_REASON'] = df_master['DISCREPANCY_REASON'].str.strip()
        
        # Return the fully processed and automated dataset
        return df_master
        
    except Exception as e:
        print(f"Workflow Automation Error: {str(e)}")
        return None

# Simulating the Drag & Drop Execution
print("Simulating User File Upload & Automated Processing...")

# Define the paths (simulating the uploaded files)
file_timesheet = r"D:\Descargas\TP AI ML Test\timesheet.csv"
file_contracts = r"D:\Descargas\TP AI ML Test\contracts.csv"
file_billing = r"D:\Descargas\TP AI ML Test\billing.csv"

# Execute the automated workflow
processed_data = automated_billing_validation(file_timesheet, file_contracts, file_billing)

if processed_data is not None:
    print("\n✅ Automated Workflow Completed Successfully.")
    print(f"Total Records: {len(processed_data)}")
    print(f"Errors Flagged: {len(processed_data[processed_data['STATUS'] == 'ERROR'])}")
    
    # Isolate errors for the AI step
    df_errors_for_ai = processed_data[processed_data['STATUS'] == 'ERROR'].copy()
    
    pd.set_option('display.max_colwidth', None)
    display(processed_data[['Employee_Name', 'Project', 'Hours_Worked', 'Hours_Billed', 'Rate_per_Hour', 'Rate_Charged', 'STATUS', 'DISCREPANCY_REASON']])

Simulating User File Upload & Automated Processing...

✅ Automated Workflow Completed Successfully.
Total Records: 5
Errors Flagged: 4


,Employee_Name,Project,Hours_Worked,Hours_Billed,Rate_per_Hour,Rate_Charged,STATUS,DISCREPANCY_REASON
0,Alice,A,40,40,20,20,OK,
1,Bob,A,38,40,20,22,ERROR,Rate Mismatch. Billed more than worked.
2,Charlie,B,45,50,25,25,ERROR,Exceeds Contract Max Hours. Billed more than worked.
3,Diana,B,42,42,25,25,ERROR,Exceeds Contract Max Hours.
4,Eve,C,36,36,30,28,ERROR,Rate Mismatch. Exceeds Contract Max Hours.


# AI System Development
Now we are going to create all the core functionalities of the AI system. 

In [28]:
import os
import getpass

# We define the route to save the API Key as a Secret Key
target_dir = r"D:\Descargas\TP AI ML Test"
env_file_path = os.path.join(target_dir, ".env")

# Here, I can safely paste the API Key for the first time in the project, it will remain safe locally.
# I want to ensure that the Key is nowhere near a public space. For the deployment, I will change it and also make it a secret key within Railway.
print("Please, paste your API Key.")
api_key = getpass.getpass()

# We create the env. archive
try:
    
    with open(env_file_path, "w") as env_file:
        env_file.write(f"OPENAI_API_KEY={api_key}\n")
    print(f"\n✅ Perfect! env. archive safely created:\n{env_file_path}")
except Exception as e:
    print(f"\n❌ Error creating the file: {e}")

Please, paste your API Key.

✅ Perfect! env. archive safely created:
D:\Descargas\TP AI ML Test\.env


In [29]:
from dotenv import load_dotenv
import os

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [36]:
import os
import json
import textwrap
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, HTML
# Adding tenacity for robust error handling
from tenacity import retry, stop_after_attempt, wait_exponential

# Environment Configuration
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("No OpenAI API key found. Please check your .env file.")

client = OpenAI(api_key=OPENAI_API_KEY)

# We use exponential backoff: it waits 4s, then 8s, up to 10s between retries.
# This is crucial for handling "API Saturated" (Rate Limit) errors.
@retry(
    stop=stop_after_attempt(3), 
    wait=wait_exponential(multiplier=1, min=4, max=10),
    reraise=True
)
def analyze_billing_errors_with_openai(df_errors):
    """
    Analyzes billing discrepancies using a language model with built-in retries.
    """
    # Selecting critical columns for audit context
    columns_for_ai = [
        'Employee_Name', 'Project', 'Hours_Worked', 'Hours_Billed', 
        'Max_Hours_Per_Week', 'Rate_per_Hour', 'Rate_Charged', 'DISCREPANCY_REASON'
    ]
    error_data_json = df_errors[columns_for_ai].to_json(orient="records")
    
    # System Prompt
    system_prompt = """
    You are an expert Financial Auditor and AI Assistant evaluating a Billing and Timesheet process.
    You MUST output your response in JSON format.
    CRITICAL: You must process EVERY single record provided. Do NOT summarize, skip, or use "..." to abbreviate. 
    The output must be exhaustive and complete for every case found in the dataset.
    """
    
    user_prompt = f"""
    I will provide you with a JSON dataset of billing records that have FAILED our mathematical validation.
    
    Here is the error dataset:
    {error_data_json}
    
    Your task is to analyze these errors and provide actionable business recommendations. 
    You MUST respond with a strictly valid JSON object using the following exact structure:
    
    {{
      "general_analysis": "Provide a 3-4 sentence executive summary of the overall error trends.",
      "fraud_risk_assessment": "Analyze if patterns indicate accidental typos or intentional manipulation. Assign a Risk Level (Low/Medium/High) and justify it.",
      "case_recommendations": [
        {{
          "Employee_Name": "Name from dataset",
          "Project": "Project from dataset",
          "ai_explanation": "Detailed natural language explanation of the specific mathematical failure.",
          "actionable_fix": "Specific action to fix the billing (e.g., credit note amount, rate adjustment)."
        }}
      ]
    }}
    """
    
    try:
        response = client.chat.completions.create(
            model="gpt-5.4", # High-precision model configuration
            response_format={ "type": "json_object" }, 
            temperature=0.2, 
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ]
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        # Tenacity will catch this and retry if it's within the attempt limit
        raise e

# Execution and Visualization Block

if 'df_errors_for_ai' in locals() and not df_errors_for_ai.empty:
    print("Sending data for analysis to the AI Model...\n")
    try:
        ai_results = analyze_billing_errors_with_openai(df_errors_for_ai)

        if "error" not in ai_results:
            # Text wrapper configuration for readability
            wrapper = textwrap.TextWrapper(width=100, break_long_words=False, replace_whitespace=False)
            separator = "=" * 100

            print(separator)
            print("AI EXECUTIVE SUMMARY")
            print(separator)
            print(wrapper.fill(ai_results.get('general_analysis', '')))
            print("\n")

            print(separator)
            print("FRAUD RISK ASSESSMENT")
            print(separator)
            print(wrapper.fill(ai_results.get('fraud_risk_assessment', '')))
            print("\n")

            print(separator)
            print("SPECIFIC CASE RECOMMENDATIONS")
            print(separator)
            
            for case in ai_results.get('case_recommendations', []):
                print(f"Employee: {case.get('Employee_Name')} | Project: {case.get('Project')}")
                print(wrapper.fill(f"Explanation: {case.get('ai_explanation')}"))
                print(wrapper.fill(f"Recommended Fix: {case.get('actionable_fix')}"))
                print("-" * 50) 
        else:
            print(f"Error: {ai_results['error']}")
    except Exception as final_e:
        print(f"API failed after multiple retries: {str(final_e)}")
else:
    print("No errors found in the dataset. AI analysis skipped.")

Sending data for analysis to the AI Model...

AI EXECUTIVE SUMMARY
The dataset shows recurring billing control failures across both hours and rate validation. Two
records include billing more hours than were worked, two records exceed contractual weekly hour
caps, and two records contain rate mismatches between the approved rate and the charged rate. The
issues are not isolated to a single project, indicating a broader process weakness in timesheet-to-
invoice reconciliation and contract rule enforcement. Overall, the trend suggests inadequate pre-
billing validation controls rather than a single one-off calculation error.


FRAUD RISK ASSESSMENT
Risk Level: Medium. The pattern includes multiple types of discrepancies, including overbilling of
hours and charging rates different from approved contract rates, which can create financial exposure
and may indicate either weak controls or deliberate revenue inflation. Because the errors affect
several employees and projects, this suggests a 

In [35]:
# I created this one without the FRAUD RISK ASSESSMENT

import os
import json
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, HTML
# Adding tenacity for robust error handling
from tenacity import retry, stop_after_attempt, wait_exponential

# Environment Configuration and Client Initialization
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("No OpenAI API key found. Please check your .env file.")

client = OpenAI(api_key=OPENAI_API_KEY)

# Configured to wait 4s, 8s, up to 10s between retries.
# Retries up to 3 times before raising the final exception.
@retry(
    stop=stop_after_attempt(3), 
    wait=wait_exponential(multiplier=1, min=4, max=10),
    reraise=True
)
def analyze_billing_errors_with_openai(df_errors):
    """
    Analyzes mathematical discrepancies and generates strategic recommendations
    using a language model with structured JSON output and built-in retries.
    """
    # Selecting critical columns to optimize context and token consumption
    columns_for_ai = [
        'Employee_Name', 'Project', 'Hours_Worked', 'Hours_Billed', 
        'Max_Hours_Per_Week', 'Rate_per_Hour', 'Rate_Charged', 'DISCREPANCY_REASON'
    ]
    error_data_json = df_errors[columns_for_ai].to_json(orient="records")
    
    system_prompt = """
    You are an expert Financial Auditor and AI Assistant evaluating a Billing and Timesheet process.
    You MUST output your response in JSON format.
    """
    
    user_prompt = f"""
    I will provide you with a JSON dataset of billing records that have FAILED our mathematical validation.
    
    Error Dataset:
    {error_data_json}
    
    Task: Analyze these errors and provide actionable business recommendations. 
    Constraint: You MUST respond with a strictly valid JSON object using this exact structure:
    
    {{
      "general_analysis": "Executive summary of error trends and high-level prevention strategy.",
      "case_recommendations": [
        {{
          "Employee_Name": "Name from dataset",
          "Project": "Project from dataset",
          "ai_explanation": "Clear explanation of the mathematical failure.",
          "actionable_fix": "Specific action to fix the billing (e.g., credit note amount, rate adjustment)."
        }}
      ]
    }}
    """
    
    try:
        response = client.chat.completions.create(
            model="gpt-5.4", # High-precision model configuration
            response_format={ "type": "json_object" },
            temperature=0.2, # Low temperature to ensure analytical consistency
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ]
        )
        
        return json.loads(response.choices[0].message.content)
        
    except Exception as e:
        # Tenacity requires the exception to be raised to trigger a retry
        raise e

#  Execution and Visualization Block 

# Verify data existence before calling the API
if 'df_errors_for_ai' in locals() and not df_errors_for_ai.empty:
    print("Sending data for analysis to the AI Model...\n")
    try:
        ai_results = analyze_billing_errors_with_openai(df_errors_for_ai)

        if "error" not in ai_results:
            separator = "=" * 80
            
            print(separator)
            print("AI EXECUTIVE SUMMARY")
            print(separator)
            print(ai_results.get('general_analysis', 'No summary provided.'))
            print("\n")
            
            print(separator)
            print("SPECIFIC CASE RECOMMENDATIONS")
            print(separator)
            
            for case in ai_results.get('case_recommendations', []):
                print(f"Employee: {case.get('Employee_Name')} | Project: {case.get('Project')}")
                print(f"Explanation: {case.get('ai_explanation')}")
                print(f"Recommended Fix: {case.get('actionable_fix')}")
                print("-" * 40)
        else:
            print(f"Analysis Error: {ai_results['error']}")
            
    except Exception as final_e:
        print(f"API request failed after all retry attempts: {str(final_e)}")
else:
    print("No errors found in the dataset. AI analysis skipped.")

Sending data for analysis to the AI Model...

AI EXECUTIVE SUMMARY
The failed records show three recurring control issues: (1) hours billed exceed hours actually worked, (2) hours worked and/or billed exceed contractual weekly maximums, and (3) billed rates do not match approved contract rates. These errors indicate weaknesses in pre-invoice validation, timesheet approval controls, and rate master governance. A high-level prevention strategy should include automated system rules that block billing above worked hours, flag any entry above contractual weekly caps, and enforce approved rate tables at invoice generation. In addition, weekly supervisor review and finance reconciliation should be required before invoices are released.


SPECIFIC CASE RECOMMENDATIONS
Employee: Bob | Project: A
Explanation: Bob worked 38 hours but 40 hours were billed, creating an overbilling of 2 hours. The approved rate is 20 per hour, but 22 per hour was charged, creating a rate overcharge of 2 per hour. Co